# getting ready

In [37]:
import importlib
import common

importlib.reload(common)

from common import *

In [2]:

export_date = datetime.now(timezone.utc).date()
date_string = export_date.strftime("%m_%d_%Y")

movie_export_url = (
    f"https://files.tmdb.org/p/exports/"
    f"movie_ids_{date_string}.json.gz"
)

print(movie_export_url)


https://files.tmdb.org/p/exports/movie_ids_11_15_2026.json.gz


In [3]:
response = requests.get(movie_export_url,stream=True,timeout=(20,180),headers={"Accept-Encoding": "identity"})

response.raise_for_status()

print(f'Status code: {response.status_code}')
print(f'Content type: {response.headers.get('Content-type')}')
print(f'Status code: {response.headers.get('Content-length')}')

Status code: 200
Content type: application/octet-stream
Status code: 28316900


Status code: 200
Content type: application/octet-stream
Status code: 28298356

In [4]:

with gzip.GzipFile(fileobj=response.raw) as gz:
    for i, line in enumerate(gz):
        record = json.loads(line)

        print(record)

        if i == 4:
            break


{'adult': False, 'id': 3924, 'original_title': 'Blondie', 'popularity': 1.0998, 'video': False}
{'adult': False, 'id': 8773, 'original_title': "L'Amour à vingt ans", 'popularity': 1.129, 'video': False}
{'adult': False, 'id': 25449, 'original_title': 'New World Disorder 9: Never Enough', 'popularity': 0.4611, 'video': False}
{'adult': False, 'id': 31975, 'original_title': 'Sesame Street: Elmo Loves You!', 'popularity': 0.1092, 'video': True}
{'adult': False, 'id': 2, 'original_title': 'Ariel', 'popularity': 1.9214, 'video': False}


{'adult': False, 'id': 3924, 'original_title': 'Blondie', 'popularity': 0.936, 'video': False}
{'adult': False, 'id': 8773, 'original_title': "L'Amour à vingt ans", 'popularity': 1.1889, 'video': False}
{'adult': False, 'id': 25449, 'original_title': 'New World Disorder 9: Never Enough', 'popularity': 0.48, 'video': False}
{'adult': False, 'id': 31975, 'original_title': 'Sesame Street: Elmo Loves You!', 'popularity': 0.0757, 'video': True}
{'adult': False, 'id': 2, 'original_title': 'Ariel', 'popularity': 1.8002, 'video': False}

In [5]:
print(f'response closed: {response.raw.closed}')

response closed: False


In [6]:

response.close()

print(f"Response closed: {response.raw.closed}")


Response closed: True


# trying movies

In [7]:

response = requests.get(
    movie_export_url,
    stream=True,
    timeout=(20, 180),
    headers={"Accept-Encoding": "identity",
             "User-Agent": "CineMatch/1.0"}
)

response.raise_for_status()

print("Connection:", response.status_code)


Connection: 200


In [8]:
total_record = 0
unique_ids = set()
duplicate_ids = 0
adult_record = 0
missing_ids = 0

In [9]:
with gzip.GzipFile(fileobj=response.raw) as gz:
    for line in gz:
        record = json.loads(line)

        total_record +=1

        movie_id = record.get('id')

        if movie_id is None:
            missing_ids +=1


        elif movie_id in unique_ids:
            duplicate_ids +=1

        else:
            unique_ids.add(movie_id)

        if record.get('adult') is True:
            adult_record +=1

response.close()


In [10]:

print(f"Total records: {total_record:,}")
print(f"Unique IDs: {len(unique_ids):,}")
print(f"Duplicate IDs: {duplicate_ids:,}")
print(f"Adult records: {adult_record:,}")
print(f"Missing IDs: {missing_ids:,}")


Total records: 1,275,981
Unique IDs: 1,275,981
Duplicate IDs: 0
Adult records: 0
Missing IDs: 0


Total records: 1,275,504
Unique IDs: 1,275,504
Duplicate IDs: 0
Adult records: 0
Missing IDs: 0

# trying tv series

In [11]:
tv_export_url = (
    f"https://files.tmdb.org/p/exports/"
    f"tv_series_ids_{date_string}.json.gz"
)

In [12]:
print(tv_export_url)

https://files.tmdb.org/p/exports/tv_series_ids_11_15_2026.json.gz


In [17]:
tv_response = requests.get(
    tv_export_url,
    stream=True,
    timeout=(20, 180),
    headers={"Accept-Encoding": "identity",
             "User-Agent": "CineMatch/1.0"})


tv_response.raise_for_status()

print(f"Status code: {tv_response.status_code}")
print(f"Content length: {tv_response.headers.get('Content-Length')}")

Status code: 200
Content length: 5166432


In [14]:
with gzip.GzipFile(fileobj=tv_response.raw) as gz:
    for i, line in enumerate(gz):
        record = json.loads(line)
        print(record)


        if i ==4:
            break

{'id': 1, 'original_name': 'プライド', 'popularity': 4.215}
{'id': 2, 'original_name': 'Clerks', 'popularity': 6.3099}
{'id': 3, 'original_name': 'The Message', 'popularity': 0.5436}
{'id': 4, 'original_name': 'The Amazing Mrs Pritchard', 'popularity': 3.1276}
{'id': 5, 'original_name': 'La Job', 'popularity': 1.4475}


{'id': 1, 'original_name': 'プライド', 'popularity': 4.3483}
{'id': 2, 'original_name': 'Clerks', 'popularity': 6.6946}
{'id': 3, 'original_name': 'The Message', 'popularity': 0.6214}
{'id': 4, 'original_name': 'The Amazing Mrs Pritchard', 'popularity': 3.027}
{'id': 5, 'original_name': 'La Job', 'popularity': 1.5998}

In [18]:
tv_total_records = 0
tv_unique_ids = set()
tv_duplicate_ids = 0
tv_missing_ids = 0

In [19]:

with gzip.GzipFile(fileobj=tv_response.raw) as gz:
    for line in gz:
        record = json.loads(line)

        tv_total_records += 1

        tv_id = record.get("id")

        if tv_id is None:
            tv_missing_ids += 1

        elif tv_id in tv_unique_ids:
            tv_duplicate_ids += 1

        else:
            tv_unique_ids.add(tv_id)

tv_response.close()


In [20]:

print(f"Total TV records: {tv_total_records:,}")
print(f"Unique TV IDs: {len(tv_unique_ids):,}")
print(f"Duplicate TV IDs: {tv_duplicate_ids:,}")
print(f"Missing TV IDs: {tv_missing_ids:,}")


Total TV records: 236,834
Unique TV IDs: 236,834
Duplicate TV IDs: 0
Missing TV IDs: 0


Total TV records: 236,734
Unique TV IDs: 236,734
Duplicate TV IDs: 0
Missing TV IDs: 0

# Sample testing

## Sample for movies

In [38]:
sample_movie_ids = list(unique_ids)[:5]

print(sample_movie_ids)

[2, 3, 5, 6, 8]


In [39]:
movie_id = sample_movie_ids[0]

movie_url = f"https://api.themoviedb.org/3/movie/{movie_id}"

In [40]:
movie_response = requests.get(movie_url,
                              headers={
        "Authorization": f"Bearer {TMDB_API_TOKEN}",
        "accept": "application/json"
    },
    timeout=(20, 60)
)

print("Status code:", movie_response.status_code)
print(movie_response.json())

Status code: 200
{'adult': False, 'backdrop_path': '/hQ4pYsIbP22TMXOUdSfC2mjWrO0.jpg', 'belongs_to_collection': None, 'budget': 0, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}, {'id': 80, 'name': 'Crime'}], 'homepage': '', 'id': 2, 'imdb_id': 'tt0094675', 'origin_country': ['FI'], 'original_language': 'fi', 'original_title': 'Ariel', 'overview': 'A Finnish man goes to the city to find a job after the mine where he worked is closed and his father commits suicide.', 'popularity': 1.9214, 'poster_path': '/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg', 'production_companies': [{'id': 2303, 'logo_path': None, 'name': 'Villealfa Filmproductions', 'origin_country': 'FI'}], 'production_countries': [{'iso_3166_1': 'FI', 'name': 'Finland'}], 'release_date': '1988-10-21', 'revenue': 0, 'runtime': 73, 'softcore': False, 'spoken_languages': [{'english_name': 'Finnish', 'iso_639_1': 'fi', 'name': 'suomi'}], 'status': 'Released', 'tagline': '', 'title': 'Ariel', 'video': False, 'vote_a

In [42]:
movie_data = movie_response.json()

print(f'Number of fields: {len(movie_data)}')
print(f'\nAvailable fields:')

for field in movie_data.keys():
    print(field)

Number of fields: 27

Available fields:
adult
backdrop_path
belongs_to_collection
budget
genres
homepage
id
imdb_id
origin_country
original_language
original_title
overview
popularity
poster_path
production_companies
production_countries
release_date
revenue
runtime
softcore
spoken_languages
status
tagline
title
video
vote_average
vote_count


In [43]:
null_fields = [field
               for field, value in movie_data.items()
               if value is None]

print(f'Total fields: {len(movie_data)}')
print(f'Null fields: {len(null_fields)}')
print(f'Null field names: {null_fields}')

Total fields: 27
Null fields: 1
Null field names: ['belongs_to_collection']


In [44]:
movie_results = []

for movie_id in sample_movie_ids:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"

    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TMDB_API_TOKEN}"},
        timeout=60
    )

    data = response.json()

    movie_results.append({
        "id": movie_id,
        "title": data.get("title"),
        "status_code": response.status_code,
        "total_fields": len(data),
        "null_fields": sum(value is None for value in data.values())
    })

pd.DataFrame(movie_results)

,id,title,status_code,total_fields,null_fields
0,2,Ariel,200,27,1
1,3,Shadows in Paradise,200,27,1
2,5,Four Rooms,200,27,1
3,6,Judgment Night,200,27,1
4,8,Life in Loops (A Megacities RMX),200,27,2


## Sample for tv_series

In [46]:
sample_tv_ids = list(tv_unique_ids)[:5]

print(sample_tv_ids)

[1, 2, 3, 4, 5]


In [47]:
tv_id = sample_tv_ids[0]

tv_url =  f"https://api.themoviedb.org/3/tv/{tv_id}"

In [49]:
tv_response = requests.get(tv_url,
                           headers={"Authorization": f"Bearer {TMDB_API_TOKEN}"},
                           timeout=60)

print(f'Status code: {tv_response.status_code}')

tv_data = tv_response.json()

print(f'Number of firlds: {len(tv_data)}')
print(list(tv_data.keys()))

Status code: 200
Number of firlds: 33
['adult', 'backdrop_path', 'created_by', 'episode_run_time', 'first_air_date', 'genres', 'homepage', 'id', 'in_production', 'languages', 'last_air_date', 'last_episode_to_air', 'name', 'next_episode_to_air', 'networks', 'number_of_episodes', 'number_of_seasons', 'origin_country', 'original_language', 'original_name', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'seasons', 'softcore', 'spoken_languages', 'status', 'tagline', 'type', 'vote_average', 'vote_count']


In [54]:
tv_results = []

for tv_id in sample_tv_ids:
    url = f"https://api.themoviedb.org/3/tv/{tv_id}"

    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {TMDB_API_TOKEN}"},
        timeout= 60
    )

    data_tv = response.json()

    tv_results.append({
            "id": tv_id,
            "name": data_tv.get("name"),
            "status_code": response.status_code,
            "total_fields": len(data_tv),
            "null_fields": sum(value is None for value in data_tv.values())
        })

pd.DataFrame(tv_results)

,id,name,status_code,total_fields,null_fields
0,1,Pride,200,33,1
1,2,Clerks,200,33,1
2,3,The Message,200,33,2
3,4,The Amazing Mrs Pritchard,200,33,1
4,5,La Job,200,33,2


In [55]:
invalid_movie_url = "https://api.themoviedb.org/3/movie/999999999"

invalid_response = requests.get(
    invalid_movie_url,
    headers={"Authorization": f"Bearer {TMDB_API_TOKEN}"},
    timeout=60
)

print("Status code:", invalid_response.status_code)
print(invalid_response.json())

Status code: 404
{'success': False, 'status_code': 34, 'status_message': 'The resource you requested could not be found.'}
